# GitHub Repository RAG

### Retrieval-Augmented Generation (RAG) system for querying and interacting with the contents of GitHub repositories.

---

**Author:** David Fernández Martínez  
**Email:** [david.fernxndez.martinez@gmail.com](mailto:david.fernxndez.martinez@gmail.com)  
**LinkedIn:** [linkedin.com/in/david-fernández-martínez](https://linkedin.com/in/david-fernández-martínez)  
**GitHub:** [github.com/davidfernxndez](https://github.com/davidfernxndez)

---

## 02 · Notebook Processing

This notebook demonstrates the **processing stage** of the RAG pipeline, focusing on the specific handling of Jupyter Notebook (`.ipynb`) files.

During the ingestion stage, repository files are initially loaded as raw text. However, some file formats require additional processing to extract only the information that is relevant to the RAG pipeline. For Jupyter Notebooks, this involves extracting the content of the notebook cells while discarding structural and metadata information that does not contribute to the retrieval process.

The implementation of the processing stage is located in `src/github_rag/processing/`:

* `processor.py`: Implements the `Processor` class, which orchestrates the different file-specific processors. This architecture allows additional processors to be easily integrated as the project evolves.
* `notebook.py`: Implements the specific processing logic for Jupyter Notebook (`.ipynb`) files.

**Notebook objectives**

* Inspect how Jupyter Notebook content is initially stored during the ingestion stage.
* Use the notebook processor to extract the relevant cell content and convert it into LangChain `Document` objects.
* Inspect the resulting documents and their associated metadata.

## Imports and Configuration

In [1]:
from github_rag.ingestion.repository import GitHubRepositoryLoader
from github_rag.processing.notebook import NotebookProcessor

You can explore any GitHub repository simply by providing its URL. For this demo, I'll be using one of my own end-to-end Data Science projects, which combines XGBoost and SHAP to build a predictive and explainable system for classifying urban typologies.

In [2]:
REPOSITORY_URL = "https://github.com/davidfernxndez/urban-typology-xai"

## 1. Notebook files format before ingestion phase

This section show how the content of .ipynb files is stored in the LangChain documents provided by the ingestion phase (`GitHubRepositoryLoader` class).

In [4]:
# Set the Loader Object with the URL
loader = GitHubRepositoryLoader(REPOSITORY_URL)

# Load the repository files as a list of LangChain Documents
documents = loader.load()

# Filter only documents of .ipynb file type
documents_ipynb = [doc for doc in documents if doc.metadata["file_type"] == ".ipynb"]

print(f"Number of .ipynb documents: {len(documents_ipynb)}")

Number of .ipynb documents: 9


In [7]:
# Select the first .ipynb document to explore
doc = documents_ipynb[0]
print(f"repository: {doc.metadata['repository']}")
print(f"File: {doc.metadata['file_path']}")
print(f"Type: {doc.metadata['file_type']}")
print(f"Source: {doc.metadata['source']}")
print(f"branch: {doc.metadata['branch']}")
print(f"Content preview:\n{doc.page_content[:2000]}")

repository: davidfernxndez/urban-typology-xai
File: notebooks/1_EDA.ipynb
Type: .ipynb
Source: https://github.com/davidfernxndez/urban-typology-xai/blob/main/notebooks/1_EDA.ipynb
branch: main
Content preview:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "a0455ca3",
   "metadata": {},
   "source": [
    "# Trabajo Fin de Máster  \n",
    "### Inteligencia artificial explicable para el estudio de la fragmentación urbana  \n",
    "#### Análisis comparativo del rendimiento y la interpretabilidad de modelos de aprendizaje supervisado para la clasificación de patrones de cerramiento residencial\n",
    "\n",
    "**Master Universitario en Ciencia de Datos e Ingeniería de Computadores (Universidad de Granada)**"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "9f8934bc",
   "metadata": {},
   "source": [
    "> **Autor:** David Fernández Martínez    \n",
    "> **Email personal:** david.fernxndez.martinez@gmail.com  \n",
    "> **Email académico:** davidfm8@correo.ugr.es  \n",


It can be observed that the notebook content is in JSON format and includes a series of fields and metadata that do not provide useful information. For RAG, we essentially need the cell content—both Markdown and code.

## 2. Notebook Processing

The `NotebookProcessor` takes a LangChain `Document` containing the raw content of a Jupyter Notebook (`.ipynb`) and transforms it into multiple `Document` objects, one for each non-empty cell in the notebook.

Cell outputs are not included, as they are considered execution results rather than part of the source code or documentation that should be retrieved by the RAG pipeline.

For each generated `Document`, the processor adds metadata describing the origin and structure of the extracted cell:
* `source_file_type`: Original file type (`.ipynb`).
* `file_type`: Content type of the cell, such as `markdown` or the corresponding programming language extension (e.g., `.py`).
* `cell_index`: Original position of the cell within the notebook.
* `cell_type`: Type of the notebook cell (`markdown` or `code`).

In [17]:
# Apply the processor for the selected document
processor = NotebookProcessor()

processed_doc = processor.process(doc)

print(f"Number of generated documents before processing: {len(processed_doc)}")

Number of generated documents before processing: 80


The notebook contains 80 non-empty cells. We can inspect some of them:

In [22]:
markdown_documents = [doc for doc in processed_doc if doc.metadata['cell_type']=="markdown"]
for i, doc in enumerate(markdown_documents[:5]):
    print(f"--- Document {i} ---")
    
    print(f"repository: {doc.metadata['repository']}")
    print(f"File: {doc.metadata['file_path']}")
    print(f"Source file type: {doc.metadata['source_file_type']}")
    print(f"Cell file type: {doc.metadata['file_type']}")
    print(f"Cell type: {doc.metadata['cell_type']}")
    print(f"Cell Index: {doc.metadata['cell_index']}")
    print(f"Source: {doc.metadata['source']}")
    print(f"branch: {doc.metadata['branch']}")
    print(f"Content preview:\n{doc.page_content[:2000]}")

--- Document 0 ---
repository: davidfernxndez/urban-typology-xai
File: notebooks/1_EDA.ipynb
Source file type: .ipynb
Cell file type: .md
Cell type: markdown
Cell Index: 0
Source: https://github.com/davidfernxndez/urban-typology-xai/blob/main/notebooks/1_EDA.ipynb
branch: main
Content preview:
# Trabajo Fin de Máster  
### Inteligencia artificial explicable para el estudio de la fragmentación urbana  
#### Análisis comparativo del rendimiento y la interpretabilidad de modelos de aprendizaje supervisado para la clasificación de patrones de cerramiento residencial

**Master Universitario en Ciencia de Datos e Ingeniería de Computadores (Universidad de Granada)**
--- Document 1 ---
repository: davidfernxndez/urban-typology-xai
File: notebooks/1_EDA.ipynb
Source file type: .ipynb
Cell file type: .md
Cell type: markdown
Cell Index: 1
Source: https://github.com/davidfernxndez/urban-typology-xai/blob/main/notebooks/1_EDA.ipynb
branch: main
Content preview:
> **Autor:** David Fernández Martíne

In [24]:
code_documents = [doc for doc in processed_doc if doc.metadata['cell_type']=="code"]
for i, doc in enumerate(code_documents[:5]):
    print(f"--- Document {i} ---")
    
    print(f"repository: {doc.metadata['repository']}")
    print(f"File: {doc.metadata['file_path']}")
    print(f"Source file type: {doc.metadata['source_file_type']}")
    print(f"Cell file type: {doc.metadata['file_type']}")
    print(f"Cell type: {doc.metadata['cell_type']}")
    print(f"Cell Index: {doc.metadata['cell_index']}")
    print(f"Source: {doc.metadata['source']}")
    print(f"branch: {doc.metadata['branch']}")
    print(f"Content preview:\n{doc.page_content[:2000]}")
    print("\n")
    print("-"*30)

--- Document 0 ---
repository: davidfernxndez/urban-typology-xai
File: notebooks/1_EDA.ipynb
Source file type: .ipynb
Cell file type: .py
Cell type: code
Cell Index: 4
Source: https://github.com/davidfernxndez/urban-typology-xai/blob/main/notebooks/1_EDA.ipynb
branch: main
Content preview:
# jupyter extensions to automatically reload external modules
%load_ext autoreload
%autoreload 2


------------------------------
--- Document 1 ---
repository: davidfernxndez/urban-typology-xai
File: notebooks/1_EDA.ipynb
Source file type: .ipynb
Cell file type: .py
Cell type: code
Cell Index: 5
Source: https://github.com/davidfernxndez/urban-typology-xai/blob/main/notebooks/1_EDA.ipynb
branch: main
Content preview:
import os
import warnings
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Configuration object
from src.config import cfg

# Utility functions for Exploratory Data Analysis (EDA)
from src.EDA_utils import (
    plot_individual

With this processing step, `.ipynb` files are no longer passed to the chunking stage as raw JSON. Instead, each non-empty cell is converted into either Markdown or source code, allowing it to be processed by the corresponding chunking strategy optimized for RAG retrieval. The original metadata is also preserved and extended to maintain traceability back to the source notebook and the specific cell from which each document was extracted.